In [51]:
require(data.table)
require(tidyverse)
require(dplyr)
require(phyloseq)
require(ggplot2)
require(RColorBrewer)
require(metacoder)
require(vegan)
require(DESeq2)
options(repr.plot.width=20, repr.plot.height=15)

## load taxa csv from symportal

In [52]:
tax=read.csv("/scratch4/workspace/caroline_desouza_uml_edu-microbe-Run1/symportal_all/its2_type_profiles/715_20260613T084002_DBV_20260614T104201.profiles.meta_only.txt", sep = '\t',header = T)

In [53]:
nrow(tax)

[1] 257

## load asv from symportal

In [54]:
asv_all=fread("/scratch4/workspace/caroline_desouza_uml_edu-microbe-Run1/symportal_all/its2_type_profiles/715_20260613T084002_DBV_20260614T104201.profiles.absolute.abund_only.txt")

In [55]:
ncol(asv_all)
head(asv_all)

[1] 258

sample_uid,2127725,2127449,2127776,2127296,2127758,2127760,2135884,2127728,2127766,⋯,2133817,2134340,2134451,2134452,2134972,2134002,2135463,2135550,2135480,2135536
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
272372,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
272591,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
272331,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
272321,0,0,0,0,0,220,0,0,0,⋯,0,3894,0,0,0,0,0,0,0,0
272281,0,0,0,0,0,2126,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
272392,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [56]:
#are there any samples with 0 across all symbint profiles
any(rowSums(asv_all != 0) == 0)

[1] FALSE

## load uid table

In [57]:
uid=fread("/scratch4/workspace/caroline_desouza_uml_edu-microbe-Run1/symportal_all/post_med_seqs/715_20260613T084002_DBV_20260614T104201.seqs.absolute.meta_only.txt", header=TRUE)

### clean up uid 

In [58]:
# Rename a single column
setnames(uid, "sample_name", "Tubelabel_species")

uid_clean <- uid[, c("sample_uid", "Tubelabel_species", "data_set_uid", "host_family", "host_genus", "host_species", "collection_date", "collection_depth")]
head(uid_clean)

sample_uid,Tubelabel_species,data_set_uid,host_family,host_genus,host_species,collection_date,collection_depth
<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>
272372,112023_BEL_CBC_T3_353_MCAV,1503,Montastraeidae,Montastrea,cavernosa,20231113,6
272591,012024_BEL_CBC_T3_640_MCAV,1503,Montastraeidae,Montastraea,cavernosa,20240110,6
272331,042024_BEL_CBC_T4_1045_MCAV,1503,Montastraeidae,Montastrea,cavernosa,20240426,6
272321,022024_BEL_CBC_T4_873_MCAV,1503,Montastraeidae,Montastrea,cavernosa,20240224,6
272281,112023_BEL_CBC_T4_399_MCAV,1503,Montastraeidae,Montastrea,cavernosa,20231113,5
272392,122023_BEL_CBC_T3_529_MCAV,1503,Montastraeidae,Montastrea,cavernosa,20231216,6


In [59]:
#make sure all sample names are separated by underscores not dashes
uid_clean$Tubelabel_species <- gsub("-", "_", uid_clean$Tubelabel_species)

## load sample data 

In [60]:
meta=fread("/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/BEL_16S_ITS2/sample_062026_PCR.csv", header=TRUE)

In [61]:
nrow(meta)

[1] 479

In [62]:
class(meta)

[1] "data.table" "data.frame"

### clean sample data

In [63]:
#cleaning samdt 
meta <- meta[, c("X" ) := NULL]
# Create a new data frame with the species, date, and transect columns
# Example samdt (make sure your data is a data.table)
setDT(meta)

### make sure uid and meta have same smaple names
- 277_OFAV_OANN in meta "112023_BEL_CBC_T1_277_OANN" in uid
- T1_63_OFAV in meta "052022_BEL_CBC_T1_63_OANN" in uid

In [64]:
### uid and meta should have 479 samples
# Keep rows in df1_clean whose 'id' does not appear in df2_clean
missing_samples <- anti_join(meta, uid_clean, by = "Tubelabel_species")
missing_samples

V1,Tubelabel_species,Date_Extracted,Raw_ng_ul,Date_Enriched,Microbe_Location,Month_year,colony,CollectionDate,Species,Health_status,Sample_physical_location,Extraction_physical_location,immune_y.n,Condition,Date_16S,Date_ITS2,Seq_run,Transect
<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>
347,112023_BEL_CBC_T1_277_OFAV_OANN,11_13_2024,7.88,,,112023,1_25,11/14/23,OANN,Bleached_Tissue,UML_NARWHAL_R5_B19,UML_NARWHAL_R2_B12,y,CLB,1_7_2026,1_19_2026,3,CBC30N
131,052022_BEL_CBC_T1_63_OFAV,7_16_2024,9.86,8_6_2024,UML_NARWHAL_R6_B27,52022,1_25,5/21/22,OANN,Healthy,UML_NARWHAL_R1_B3,UML_NARWHAL_R2_B26,y,Healthy,1_8_2026,1_20_2026,3,CBC30N


In [65]:
uid_clean <- uid_clean %>%
  mutate(Tubelabel_species = case_match(
    Tubelabel_species,
    "112023_BEL_CBC_T1_277_OANN" ~ "112023_BEL_CBC_T1_277_OFAV_OANN",
    "052022_BEL_CBC_T1_63_OANN"  ~ "052022_BEL_CBC_T1_63_OFAV",
    .default = Tubelabel_species # Keeps all other sample names exactly as they were
  ))

### merge meta with uid

In [66]:
# Merge by the common column "ID" (default is inner join)
meta <- merge(meta, uid_clean, by = "Tubelabel_species")
head(meta)
nrow(meta)

Tubelabel_species,V1,Date_Extracted,Raw_ng_ul,Date_Enriched,Microbe_Location,Month_year,colony,CollectionDate,Species,⋯,Date_ITS2,Seq_run,Transect,sample_uid,data_set_uid,host_family,host_genus,host_species,collection_date,collection_depth
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,⋯,<chr>,<int>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<int>,<int>
012024_BEL_CBC_T1_557_SSID,1,2_17_2025,44.6,,,12024,1_3,1/10/24,SSID,⋯,7_7_2025,1,CBC30N,272206,1503,Rhizangiidae,Siderastrea,sidera,20240110,8
012024_BEL_CBC_T1_559_MCAV,2,2_17_2025,14.8,,,12024,1_24,1/10/24,MCAV,⋯,12_15_2025,2,CBC30N,272302,1503,Montastraeidae,Montastrea,cavernosa,20240110,8
012024_BEL_CBC_T1_561_OANN,3,2_17_2025,18,,,12024,1_25,1/10/24,OANN,⋯,1_21_2026,3,CBC30N,272398,1503,Merulinidae,Orbicella,annularis,20240110,8
012024_BEL_CBC_T1_563_PSTR,4,2_17_2025,35.8,,,12024,1_12,1/10/24,PSTR,⋯,7_7_2025,1,CBC30N,272207,1503,Faviidae,Pseudodiploria,strigosa,20240110,8
012024_BEL_CBC_T1_565_PAST,5,2_17_2025,7.72,,,12024,1_21,1/10/24,PAST,⋯,2_17_2026,4,CBC30N,272494,1503,Poritidae,Porites,asteroides,20240110,8
012024_BEL_CBC_T2_585_OFAV,6,2_17_2025,71.6,,,12024,2_76,1/12/24,OFAV,⋯,2_17_2026,4,SR30N,272495,1503,Merulinidae,Orbicella,faveolata,20240112,8


[1] 479

In [67]:
# align rownames and columns
#optional to set rownames
metadf <- as.data.frame(meta)              # Convert to data.frame (from data.table)
rownames(metadf) <- metadf$Tubelabel_species # Set rownames
metadf$Tubelabel_species <- NULL  

### chronological order dates

In [68]:
#changing format of Month_year to be Jan 2024 using collection date
metadf$CollectionDate <- as.factor(metadf$CollectionDate)
# Convert to Date format 
metadf$DateFormatted <- as.Date(paste0(metadf$CollectionDate), format = "%m/%d/%y")
# now month_year will be organized by chronological date
metadf$Month_year <- format(metadf$DateFormatted, "%b %Y")
metadf$Month_year <- factor(metadf$Month_year, levels = unique(metadf$Month_year))
head(metadf)

,V1,Date_Extracted,Raw_ng_ul,Date_Enriched,Microbe_Location,Month_year,colony,CollectionDate,Species,Health_status,⋯,Seq_run,Transect,sample_uid,data_set_uid,host_family,host_genus,host_species,collection_date,collection_depth,DateFormatted
,<int>,<chr>,<chr>,<chr>,<chr>,<fct>,<chr>,<fct>,<chr>,<chr>,⋯,<int>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<date>
012024_BEL_CBC_T1_557_SSID,1,2_17_2025,44.6,,,Jan 2024,1_3,1/10/24,SSID,Healthy,⋯,1,CBC30N,272206,1503,Rhizangiidae,Siderastrea,sidera,20240110,8,2024-01-10
012024_BEL_CBC_T1_559_MCAV,2,2_17_2025,14.8,,,Jan 2024,1_24,1/10/24,MCAV,Healthy,⋯,2,CBC30N,272302,1503,Montastraeidae,Montastrea,cavernosa,20240110,8,2024-01-10
012024_BEL_CBC_T1_561_OANN,3,2_17_2025,18,,,Jan 2024,1_25,1/10/24,OANN,Healthy,⋯,3,CBC30N,272398,1503,Merulinidae,Orbicella,annularis,20240110,8,2024-01-10
012024_BEL_CBC_T1_563_PSTR,4,2_17_2025,35.8,,,Jan 2024,1_12,1/10/24,PSTR,Healthy,⋯,1,CBC30N,272207,1503,Faviidae,Pseudodiploria,strigosa,20240110,8,2024-01-10
012024_BEL_CBC_T1_565_PAST,5,2_17_2025,7.72,,,Jan 2024,1_21,1/10/24,PAST,Bleached_Tissue,⋯,4,CBC30N,272494,1503,Poritidae,Porites,asteroides,20240110,8,2024-01-10
012024_BEL_CBC_T2_585_OFAV,6,2_17_2025,71.6,,,Jan 2024,2_76,1/12/24,OFAV,Healthy,⋯,4,SR30N,272495,1503,Merulinidae,Orbicella,faveolata,20240112,8,2024-01-12


### replacing uid in asv table with tubelabel_species

In [111]:
#new df with just sample_uid with Tubelabel_species
name <- uid_clean <- uid_clean[, c("sample_uid", "Tubelabel_species")]

In [112]:
name <- as.data.frame(name)
head(name)

,sample_uid,Tubelabel_species
,<int>,<chr>
1,272372,112023_BEL_CBC_T3_353_MCAV
2,272591,012024_BEL_CBC_T3_640_MCAV
3,272331,042024_BEL_CBC_T4_1045_MCAV
4,272321,022024_BEL_CBC_T4_873_MCAV
5,272281,112023_BEL_CBC_T4_399_MCAV
6,272392,122023_BEL_CBC_T3_529_MCAV


In [113]:
head(asv_all)
asv_df <- as.data.frame(asv_all)

sample_uid,2127725,2127449,2127776,2127296,2127758,2127760,2135884,2127728,2127766,⋯,2133817,2134340,2134451,2134452,2134972,2134002,2135463,2135550,2135480,2135536
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
272372,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
272591,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
272331,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
272321,0,0,0,0,0,220,0,0,0,⋯,0,3894,0,0,0,0,0,0,0,0
272281,0,0,0,0,0,2126,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
272392,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [114]:
head(asv_df)

,sample_uid,2127725,2127449,2127776,2127296,2127758,2127760,2135884,2127728,2127766,⋯,2133817,2134340,2134451,2134452,2134972,2134002,2135463,2135550,2135480,2135536
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,272372,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
2,272591,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
3,272331,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
4,272321,0,0,0,0,0,220,0,0,0,⋯,0,3894,0,0,0,0,0,0,0,0
5,272281,0,0,0,0,0,2126,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
6,272392,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [115]:

#add tubelabel_species column in asv

# Merge by the common column "ID" (default is inner join)
asv <- merge(name, asv_df, by = "sample_uid")
head(asv)

,sample_uid,Tubelabel_species,2127725,2127449,2127776,2127296,2127758,2127760,2135884,2127728,⋯,2133817,2134340,2134451,2134452,2134972,2134002,2135463,2135550,2135480,2135536
,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,272206,012024_BEL_CBC_T1_557_SSID,0,0,0,0,0,0,469,0,⋯,0,0,0,0,0,0,0,0,0,0
2,272207,012024_BEL_CBC_T1_563_PSTR,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
3,272208,012024_BEL_CBC_T2_601_OFAV,918825,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
4,272209,012024_BEL_CBC_T2_605_SSID,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
5,272210,012024_BEL_CBC_T3_627_PAST,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
6,272211,012024_BEL_CBC_T3_631_MCAV,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [116]:
### asv will have 480 because I sequenced 480 samples, but one of those samples was AS
nrow(asv)

[1] 480

In [117]:
#remove sample_uid
asv$sample_uid <- NULL
head(asv)

,Tubelabel_species,2127725,2127449,2127776,2127296,2127758,2127760,2135884,2127728,2127766,⋯,2133817,2134340,2134451,2134452,2134972,2134002,2135463,2135550,2135480,2135536
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,012024_BEL_CBC_T1_557_SSID,0,0,0,0,0,0,469,0,0,⋯,0,0,0,0,0,0,0,0,0,0
2,012024_BEL_CBC_T1_563_PSTR,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
3,012024_BEL_CBC_T2_601_OFAV,918825,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
4,012024_BEL_CBC_T2_605_SSID,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
5,012024_BEL_CBC_T3_627_PAST,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
6,012024_BEL_CBC_T3_631_MCAV,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [118]:
#making Tubelabel_species rownames 
rownames(asv) <- asv[,1]
 asv <- asv[,-1]

In [119]:
head(asv)

,2127725,2127449,2127776,2127296,2127758,2127760,2135884,2127728,2127766,2127763,⋯,2133817,2134340,2134451,2134452,2134972,2134002,2135463,2135550,2135480,2135536
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
012024_BEL_CBC_T1_557_SSID,0,0,0,0,0,0,469,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_563_PSTR,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T2_601_OFAV,918825,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T2_605_SSID,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T3_627_PAST,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T3_631_MCAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


## make sure datatable sare compatible for phyloseq

In [120]:
#which samples have no symbionts
# are there any samples that never have symbionts?
which(rowSums(asv == 0) == ncol(asv))

# if so are these the same three samples that have 0 reads?

named integer(0)

In [124]:
# align rownames and columns
#optional to set rownames
taxadf <- as.data.frame(tax)  # Convert to data.frame (from data.table)

## make ITS2 type profile UID be rownames
rownames(taxadf) <- taxadf$ITS2.type.profile.UID
taxadf$ITS2.type.profile.UID <- NULL    
head(taxadf)

,Clade,Majority.ITS2.sequence,Associated.species,ITS2.profile.abundance.local,ITS2.profile.abundance.DB,ITS2.type.profile,Sequence.accession...SymPortal.UID,Average.defining.sequence.proportions.and..stdev.
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>
2127725,A,A3,"Symbiodinium natans,Symbiodinium tridacnidorum",30,65,A3-A3ah-A3ga,22385-36733-6483645,0.933[0.013]-0.043[0.004]-0.024[0.010]
2127449,A,A3bb/A3bt,None,29,496,A3bb/A3bt,,0.553[0.069]/0.447[0.069]
2127776,A,A3,"Symbiodinium natans,Symbiodinium tridacnidorum",18,38,A3-A4-A4a-A3ga,22385-29206-40166-6483645,0.853[0.114]-0.092[0.082]-0.036[0.029]-0.019[0.014]
2127296,A,A4/A3/A4a,"Symbiodinium natans,Symbiodinium tridacnidorum",17,51,A3/A4/A4a,,0.388[0.221]/0.369[0.352]/0.243[0.165]
2127758,A,A3/A4,"Symbiodinium natans,Symbiodinium tridacnidorum",16,35,A3/A4-A4a-A4fe-A4fo-A4fn,22385/29206-40166-318716-136549-59064,0.500[0.208]/0.263[0.124]-0.109[0.071]-0.053[0.035]-0.048[0.033]-0.028[0.016]
2127760,A,A3/A4,"Symbiodinium natans,Symbiodinium tridacnidorum",16,34,A3/A4-A4a-A4fe-A4au-A4bk,22385/29206-40166-318716-37992-29353,0.428[0.195]/0.367[0.147]-0.120[0.055]-0.041[0.023]-0.024[0.010]-0.020[0.010]


In [125]:
# rownames for asv and meta match
#make sure rownames are in same order
common_samples <- intersect(rownames(metadf), rownames(asv))

# Reorder both objects to match the same order
metadf <- metadf[common_samples, , drop = FALSE]
asv <- asv[common_samples, , drop = FALSE]

In [126]:
head(asv)

,2127725,2127449,2127776,2127296,2127758,2127760,2135884,2127728,2127766,2127763,⋯,2133817,2134340,2134451,2134452,2134972,2134002,2135463,2135550,2135480,2135536
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
012024_BEL_CBC_T1_557_SSID,0,0,0,0,0,0,469,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_559_MCAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_561_OANN,524180,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_563_PSTR,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_565_PAST,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T2_585_OFAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [127]:
## colnames for taxa and asv 

# Reorder taxa to match asv_all
#Get common ASVs
common_asvs <- intersect(colnames(asv), rownames(taxadf))

## investigate why this is removing all taxa from asv
class(common_asvs)
head(common_asvs)

[1] "character"

[1] "2127725" "2127449" "2127776" "2127296" "2127758" "2127760"

In [128]:
# Subset both
asv <- asv[, common_asvs]
taxadf <- taxadf[common_asvs, ]

In [129]:
nrow(metadf)
nrow(asv)

[1] 479

[1] 479

In [130]:
head(asv)

,2127725,2127449,2127776,2127296,2127758,2127760,2135884,2127728,2127766,2127763,⋯,2133817,2134340,2134451,2134452,2134972,2134002,2135463,2135550,2135480,2135536
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
012024_BEL_CBC_T1_557_SSID,0,0,0,0,0,0,469,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_559_MCAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_561_OANN,524180,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_563_PSTR,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T1_565_PAST,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
012024_BEL_CBC_T2_585_OFAV,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


## phyloseq

In [131]:
# final checks

#double check that all rownames match
all(rownames(asv) %in% rownames(metadf))  # Should be TRUE
all(rownames(metadf) %in% rownames(asv))  # Should be TRUE

sum(is.na(match(colnames(asv), rownames(taxadf))))
identical(colnames(asv), rownames(taxadf))  # Should be TRUE

#check if asv_all is numeric
all(sapply(as.data.table(asv), is.numeric))

identical(colnames(asv), rownames(taxadf))  # MUST be TRUE
identical(rownames(asv), rownames(metadf))  # MUST be TRUE

[1] TRUE

[1] TRUE

[1] 0

[1] TRUE

[1] TRUE

[1] TRUE

[1] TRUE

In [133]:
# Convert asv_all to otu_table (specify taxa_are_rows = TRUE/FALSE depending on your data)
otu <- otu_table(asv, taxa_are_rows = FALSE)  # or TRUE if taxa in rows

# Sample metadata as sample_data
sam <- sample_data(metadf)

taxa_mat <- as.matrix(taxadf)
profile=tax_table(taxa_mat)

# Construct phyloseq object
ps <- phyloseq(otu, sam, profile)

In [134]:
ps
saveRDS(ps, file = "/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/BEL_16S_ITS2/BEL_ITS2_outputs/ps_all_ITS2.rds")

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 257 taxa and 479 samples ]
sample_data() Sample Data:       [ 479 samples by 26 sample variables ]
tax_table()   Taxonomy Table:    [ 257 taxa by 8 taxonomic ranks ]